# 02 · Fitur teks: BERTopic, GPR, sentimen, agregasi harian, PCA, time series
Semua yang belajar (BERTopic, PCA, scaler) di-fit **hanya pada Train**.

In [3]:
import re, json
import numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from utils import PROC, CACHE, SEED, fingerprint

A = pd.read_parquet(PROC / 'articles.parquet')
daily = pd.read_parquet(PROC / 'daily.parquet').set_index('date')
meta = json.load(open(CACHE / 'meta.json'))
assert meta['fingerprint'] == fingerprint(A) and meta['n'] == len(A), 'cache embedding tidak cocok dengan tabel artikel, buat ulang'
ET, EB = np.load(CACHE / 'emb_title.npy'), np.load(CACHE / 'emb_body.npy')
A['sent'] = np.load(CACHE / 'sentiment.npy')

A['split'] = A.aligned_trading_date.map(daily.split)
tr = (A.split == 'train').values
use = A.split.isin(['train', 'val', 'test']).values
days = daily.index[daily.split != 'none']                 # hanya hari di dalam split
trd = daily.index[daily.split == 'train']
print('artikel dipakai:', use.sum(), '| train:', tr.sum())

C:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


artikel dipakai: 6563 | train: 4737


## Langkah 2: BERTopic (fit Train, Val/Test hanya transform), GPR kamus, sentimen

In [4]:
E = ET + EB; E /= np.linalg.norm(E, axis=1, keepdims=True)
docs = (A.title + '. ' + A.text).tolist()
tm = BERTopic(umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0, metric='cosine', random_state=SEED),
              hdbscan_model=HDBSCAN(min_cluster_size=50, prediction_data=True),
              vectorizer_model=CountVectorizer(min_df=5, max_df=0.5),
              nr_topics=12, calculate_probabilities=True)
tm.fit([d for d, t in zip(docs, tr) if t], E[tr])
_, P = tm.transform(docs, E)                              # distribusi probabilitas topik, bukan label keras
P = np.asarray(P)
print('topik:', P.shape[1]); print(tm.get_topic_info().head(15)[['Topic', 'Count', 'Name']].to_string())

topik: 11
    Topic  Count                                 Name
0      -1   1227  -1_masjid_greenland_uranium_starmer
1       0   1816        0_netanyahu_mojtaba_irgc_gaza
2       1    493          1_el_narkoba_gempa_kolombia
3       2    310           2_gaza_board_sugiono_hamas
4       3    245             3_paus_leo_jeffrey_email
5       4    151         4_taiwan_wang_tiongkok_nepal
6       5    145     5_kremlin_peskov_araghchi_dmitry
7       6     94          6_haji_jemaah_ramadan_puasa
8       7     79      7_relawan_aktivis_berlayar_gaza
9       8     70         8_tni_rico_iskandar_guterres
10      9     55                 9_kim_jong_korut_lee
11     10     52       10_gempa_dahsyat_246_magnitudo


In [5]:
# Kamus GPR (stem, tanpa nama negara). Ditulis sebelum melihat hasil model. REVIEW sebelum dijalankan.
GPR_STEMS = ['perang', 'militer', 'tentara', 'pasukan', 'rudal', 'drone', 'bom', 'ledak', 'senjata', 'nuklir', 'serang', 'invasi',
             'agresi', 'ofensif', 'tembak', 'tewas', 'gugur', 'konflik', 'tegang', 'ancam', 'teror', 'sanksi', 'embargo', 'blokade',
             'gencatan', 'perundingan', 'diplomat', 'aliansi', 'mobilisasi', 'kudeta', 'pengungsi', 'krisis', 'intelijen',
             'pertahanan', 'pendudukan', 'aneksasi', 'separatis', 'pemberontak', 'milisi', 'ekstremis', 'ranjau', 'gempur']
gpr_re = re.compile(r'\b(?:' + '|'.join(GPR_STEMS) + r')\w*')
low = A.text.str.lower()
A['gpr'] = low.map(lambda t: len(gpr_re.findall(t))) / low.str.split().str.len().clip(lower=1)

## Langkah 3: agregasi harian, PCA per blok (fit Train), standardisasi, hari tanpa berita = 0

In [6]:
U = A[use].copy(); U['w'] = U.gpr + 1e-6; U['sw'] = U.sent * U.w
key = U.aligned_trading_date.values
g = U.groupby('aligned_trading_date')
D = pd.DataFrame({'sent_mean': g.sent.mean(), 'sent_min': g.sent.min(), 'sent_gprw': g.sw.sum() / g.w.sum(),
                  'gpr_mean': g.gpr.mean(), 'gpr_max': g.gpr.max(), 'news_count': g.size().astype(float)})
dmean = lambda M: pd.DataFrame(M[use]).groupby(key).mean()
MT, MB = dmean(ET), dmean(EB)
TP = dmean(P); TP.columns = [f'top{i}' for i in range(TP.shape[1])]

def pca(M, pref, k=5):
    p = PCA(k, random_state=SEED).fit(M[M.index.isin(trd)].values)
    return pd.DataFrame(p.transform(M.values), index=M.index, columns=[f'{pref}{i + 1}' for i in range(k)])

V = MT.values + MB.values; V = V / np.linalg.norm(V, axis=1, keepdims=True)    # novelty: jarak kosinus ke rata-rata 5 hari berita terakhir
nov = [0.0 if i == 0 else 1 - V[i] @ V[max(0, i - 5):i].mean(0) / np.linalg.norm(V[max(0, i - 5):i].mean(0)) for i in range(len(V))]

X = pd.concat([D, pca(MT, 'emT'), pca(MB, 'emB'), TP, pd.Series(nov, index=MT.index, name='nov')], axis=1)
std = [c for c in X.columns if c != 'news_count']
X[std] = StandardScaler().fit(X.loc[X.index.isin(trd), std]).transform(X[std])
X['has_news'] = 1.0
X = X.reindex(days).fillna(0)                              # hari tanpa berita: 0 (netral) + has_news = 0
print(X.shape)

(141, 29)


## Langkah 4: fitur time series (lag, rolling, selisih, surprise). Hanya masa lalu dan hari ini.

In [7]:
def ts_feats(X, S=('sent_mean', 'gpr_mean', 'emT1', 'emB1')):
    F = X.copy()
    for c in S:
        x, past = X[c], X[c].shift(1)
        for l in (1, 2, 3): F[f'{c}_lag{l}'] = x.shift(l)
        F[f'{c}_r3'], F[f'{c}_r5'] = x.rolling(3, min_periods=1).mean(), x.rolling(5, min_periods=1).mean()
        F[f'{c}_d1'] = x - x.shift(1)
        F[f'{c}_sur'] = (x - past.expanding(min_periods=5).mean()) / past.expanding(min_periods=5).std().replace(0, np.nan)
    return F.replace([np.inf, -np.inf], np.nan).fillna(0)

F = ts_feats(X)
# uji kausal: menghapus baris terakhir tidak boleh mengubah fitur hari-hari sebelumnya
pd.testing.assert_frame_equal(ts_feats(X.iloc[:-1]), F.iloc[:-1])
F.rename_axis('date').reset_index().to_parquet(PROC / 'features_text.parquet')
print(F.shape, '| kolom teks total:', F.shape[1])

(141, 57) | kolom teks total: 57
